### CNN trained on data encoded with protein scaling factor

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from sklearn.model_selection import KFold
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score 
import json

Load the labels

In [3]:
def load_labels(file_path):
    with open(file_path, 'r') as file:
        labels = file.readlines()
    labels = np.array([int(label.strip()) for label in labels])
    return labels

Load the images and labels into arrays

In [ ]:
def load_data(image_folder, labels_file):
    labels = load_labels(labels_file)
    image_size = (124, 124)  # Resize to a smaller image size
    images = []
    
    for img_name in sorted(os.listdir(image_folder)):
        img_path = os.path.join(image_folder, img_name)
        img = tf.keras.preprocessing.image.load_img(img_path, target_size=image_size, color_mode='grayscale')
        img_array = tf.keras.preprocessing.image.img_to_array(img)
        images.append(img_array)
    
    images = np.array(images)
    return images, labels


CNN model

In [ ]:
def build_cnn_model(input_shape):
    model = Sequential([
        Input(shape=input_shape),

        Conv2D(32, kernel_size=(3, 3), kernel_regularizer=l2(0.001), activation='relu', padding='same'),
        MaxPooling2D(pool_size=(2, 2)),

        Conv2D(64, kernel_size=(3, 3), kernel_regularizer=l2(0.001), activation='relu', padding='same'),
        MaxPooling2D(pool_size=(2, 2)),

        Flatten(),

        Dense(128, activation='relu', kernel_regularizer=l2(0.001)),
        Dropout(0.5),

        Dense(1, activation='sigmoid')
    ]) 

    model.compile(optimizer=Adam(learning_rate=0.0005), loss='binary_crossentropy', metrics=['accuracy'])

    return model

Cross-validation

In [ ]:
def cross_validate_model(image_folder, labels_file, num_folds=10):
    # Load the dataset
    images, labels = load_data(image_folder, labels_file)
    
    # Normalize images
    images = images.astype('float32') / 255.0

    X_trainval, X_test, y_trainval, y_test = train_test_split(images, labels, test_size=0.2, random_state=42, stratify=labels)

    # Define KFold cross-validation on the train set
    kf = KFold(n_splits=num_folds, shuffle=True, random_state=42)

    fold_no = 1
    accuracy_per_fold = []
    auc_per_fold = []
    all_classification_reports = []
    all_auc_scores = []

    for train_index, val_index in kf.split(X_trainval):  # KFold only on train set
        X_train_fold, X_val_fold = X_trainval[train_index], X_trainval[val_index]
        y_train_fold, y_val_fold = y_trainval[train_index], y_trainval[val_index]

        # Build and compile the model
        input_shape = X_trainval.shape[1:]
        model = build_cnn_model(input_shape)

        print(f'\nTraining fold {fold_no}...')

        # Train the model
        model.fit(X_train_fold, y_train_fold , validation_data=(X_val_fold, y_val_fold), epochs=20, batch_size=8, verbose=1)

        # Evaluate on the test set
        y_pred_prob = model.predict(X_test)

        y_pred = (y_pred_prob > 0.5).astype(int)

        # Calculate accuracy
        accuracy = accuracy_score(y_test, y_pred) 
        accuracy_per_fold.append(accuracy)

        # Calculate AUC
        auc = roc_auc_score(y_test, y_pred_prob)
        auc_per_fold.append(auc)
        all_auc_scores.append(auc)

        # Classification Report
        class_report = classification_report(y_test, y_pred, target_names=['Negative', 'Positive'], output_dict=True, zero_division=0)
        all_classification_reports.append(class_report)

        print(f'Fold {fold_no} accuracy: {accuracy:.4f}')
        print(f'Fold {fold_no} AUC: {auc:.4f}')

        fold_no += 1

    # Average results
    avg_accuracy = np.mean(accuracy_per_fold)
    std_accuracy = np.std(accuracy_per_fold)
    avg_auc = np.mean(auc_per_fold)
    std_auc = np.std(auc_per_fold)

    # Average classification metrics
    avg_classification_report = {
        'Positive': {
            'accuracy': float(avg_accuracy),
            'auc': float(avg_auc),
            'precision': float(np.mean([r['Positive']['precision'] for r in all_classification_reports])),
            'recall': float(np.mean([r['Positive']['recall'] for r in all_classification_reports])),
            'f1-score': float(np.mean([r['Positive']['f1-score'] for r in all_classification_reports]))
        },
        'Negative': {
            'accuracy': float(avg_accuracy),
            'auc': float(avg_auc),
            'precision': float(np.mean([r['Negative']['precision'] for r in all_classification_reports])),
            'recall': float(np.mean([r['Negative']['recall'] for r in all_classification_reports])),
            'f1-score': float(np.mean([r['Negative']['f1-score'] for r in all_classification_reports]))
        }
    }

    print("\nAverage Classification Report (across all folds):")
    print(avg_classification_report)

    return accuracy_per_fold, auc_per_fold, all_classification_reports, avg_accuracy, std_accuracy, avg_auc, std_auc, avg_classification_report


Saving the results into a JSON

In [8]:
def save_classification_report(dataset_name, avg_classification_report):
    results_file = "reports/nosf/classification_reports3_cnn.json"

    # Add dataset name to the report
    report_to_save = {
        "dataset": dataset_name,
        "report": avg_classification_report
    }

    # Load existing results if the file exists
    if os.path.exists(results_file):
        with open(results_file, "r") as f:
            existing_results = json.load(f)
    else:
        existing_results = []

    # Append new results
    existing_results.append(report_to_save)

    # Save updated results
    with open(results_file, "w") as f:
        json.dump(existing_results, f, indent=4)

    print(f"Classification report saved to {results_file}")

In [9]:
def save_results(dataset_name, accuracy_per_fold, auc_per_fold, all_classification_reports):
    results_file = "reports/nosf/classification_results3_CNN.json"

    # Convert results to a dictionary
    results_dict = {
        "dataset": dataset_name,
        "accuracies": accuracy_per_fold,
        "roc_aucs": auc_per_fold,
        "classification_report": all_classification_reports
    }

    # Load existing results if the file exists
    if os.path.exists(results_file):
        with open(results_file, "r") as f:
            existing_results = json.load(f)
    else:
        existing_results = []

    # Append new results
    existing_results.append(results_dict)

    # Save updated results
    with open(results_file, "w") as f:
        json.dump(existing_results, f, indent=4)

    print(f"Results saved to {results_file}")



File paths RES 25

In [9]:
image_folder_antiinflam = 'data/images/img_nosf/img25/aip_antiinflam' 
labels_file_antiinflam = 'data/labels/aip_antiinflam.txt'

In [10]:
image_folder_antipb = 'data/images/img_nosf/img25/amp_antibp'
labels_file_antipb = 'data/labels/amp_antibp.txt'


In [11]:
image_folder_antipb2 = 'data/images/img_nosf/img25/amp_antibp2'
labels_file_antipb2 = 'data/labels/amp_antibp2.txt'

In [12]:
image_folder_csamp = 'data/images/img_nosf/img25/amp_csamp'
labels_file_csamp = 'data/labels/amp_csamp.txt'

In [13]:
image_folder_hivddi = 'data/images/img_nosf/img25/hiv_ddi'
labels_file_hivddi = 'data/labels/hiv_ddi.txt'

In [14]:
image_folder_hivrtv = 'data/images/img_nosf/img25/hiv_rtv'
labels_file_hivrtv = 'data/labels/hiv_rtv.txt'

TRAINING RES 25

In [15]:
accuracy_per_fold, auc_per_fold, all_classification_reports, avg_accuracy, std_accuracy, avg_auc, std_auc, avg_classification_report  = cross_validate_model(image_folder_antiinflam, labels_file_antiinflam, num_folds=10 )


Training fold 1...
Epoch 1/20
192/192 ━━━━━━━━━━━━━━━━━━━━ 24s 118ms/step - accuracy: 0.5611 - loss: 0.8451 - val_accuracy: 0.6529 - val_loss: 0.6966
Epoch 2/20
192/192 ━━━━━━━━━━━━━━━━━━━━ 25s 129ms/step - accuracy: 0.6408 - loss: 0.6883 - val_accuracy: 0.6706 - val_loss: 0.6440
Epoch 3/20
192/192 ━━━━━━━━━━━━━━━━━━━━ 23s 121ms/step - accuracy: 0.7341 - loss: 0.6090 - val_accuracy: 0.6765 - val_loss: 0.6384
Epoch 4/20
192/192 ━━━━━━━━━━━━━━━━━━━━ 23s 120ms/step - accuracy: 0.7380 - loss: 0.5788 - val_accuracy: 0.6941 - val_loss: 0.6298
Epoch 5/20
192/192 ━━━━━━━━━━━━━━━━━━━━ 23s 120ms/step - accuracy: 0.7481 - loss: 0.5530 - val_accuracy: 0.7294 - val_loss: 0.6541
Epoch 6/20
192/192 ━━━━━━━━━━━━━━━━━━━━ 23s 121ms/step - accuracy: 0.8023 - loss: 0.5201 - val_accuracy: 0.7235 - val_loss: 0.6295
Epoch 7/20
192/192 ━━━━━━━━━━━━━━━━━━━━ 23s 122ms/step - accuracy: 0.8293 - loss: 0.4992 - val_accuracy: 0.6765 - val_loss: 0.6723
Epoch 8/20
192/192 ━━━━━━━━━━━━━━━━━━━━ 23s 122ms/step - accura

In [18]:
save_results("aip_antiinflam_25", accuracy_per_fold, auc_per_fold, all_classification_reports)
save_classification_report("aip_antiinflam_25", avg_classification_report)

Results saved to reports/nosf/classification_results3_CNN.json
Classification report saved to reports/nosf/classification_reports3_cnn.json


In [19]:
accuracy_per_fold, auc_per_fold, all_classification_reports, avg_accuracy, std_accuracy, avg_auc, std_auc, avg_classification_report = cross_validate_model(image_folder_antipb, labels_file_antipb, num_folds=10 )


Training fold 1...
Epoch 1/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 10s 121ms/step - accuracy: 0.5088 - loss: 1.0653 - val_accuracy: 0.7536 - val_loss: 0.7395
Epoch 2/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 9s 117ms/step - accuracy: 0.6299 - loss: 0.7251 - val_accuracy: 0.7536 - val_loss: 0.6367
Epoch 3/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 9s 118ms/step - accuracy: 0.7237 - loss: 0.6236 - val_accuracy: 0.8406 - val_loss: 0.5549
Epoch 4/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 9s 119ms/step - accuracy: 0.7724 - loss: 0.5622 - val_accuracy: 0.8841 - val_loss: 0.4954
Epoch 5/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 9s 119ms/step - accuracy: 0.8044 - loss: 0.5083 - val_accuracy: 0.8116 - val_loss: 0.5225
Epoch 6/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 9s 118ms/step - accuracy: 0.8471 - loss: 0.4594 - val_accuracy: 0.8551 - val_loss: 0.4862
Epoch 7/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 9s 118ms/step - accuracy: 0.8627 - loss: 0.4624 - val_accuracy: 0.8406 - val_loss: 0.4697
Epoch 8/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 9s 119ms/step - accuracy: 0.8634 - loss: 0.42

In [20]:
save_results("amp_antibp_25", accuracy_per_fold, auc_per_fold, all_classification_reports)
save_classification_report("amp_antibp_25", avg_classification_report)

Results saved to reports/nosf/classification_results3_CNN.json
Classification report saved to reports/nosf/classification_reports3_cnn.json


In [21]:
accuracy_per_fold, auc_per_fold, all_classification_reports, avg_accuracy, std_accuracy, avg_auc, std_auc, avg_classification_report  = cross_validate_model(image_folder_antipb2, labels_file_antipb2, num_folds=10 )


Training fold 1...
Epoch 1/20
180/180 ━━━━━━━━━━━━━━━━━━━━ 21s 110ms/step - accuracy: 0.5395 - loss: 0.8990 - val_accuracy: 0.8062 - val_loss: 0.5669
Epoch 2/20
180/180 ━━━━━━━━━━━━━━━━━━━━ 20s 110ms/step - accuracy: 0.7508 - loss: 0.6043 - val_accuracy: 0.7750 - val_loss: 0.5263
Epoch 3/20
180/180 ━━━━━━━━━━━━━━━━━━━━ 20s 109ms/step - accuracy: 0.7777 - loss: 0.5404 - val_accuracy: 0.7812 - val_loss: 0.5220
Epoch 4/20
180/180 ━━━━━━━━━━━━━━━━━━━━ 20s 110ms/step - accuracy: 0.7764 - loss: 0.5255 - val_accuracy: 0.7812 - val_loss: 0.5195
Epoch 5/20
180/180 ━━━━━━━━━━━━━━━━━━━━ 20s 110ms/step - accuracy: 0.8180 - loss: 0.4952 - val_accuracy: 0.7875 - val_loss: 0.5005
Epoch 6/20
180/180 ━━━━━━━━━━━━━━━━━━━━ 20s 111ms/step - accuracy: 0.8386 - loss: 0.4411 - val_accuracy: 0.7937 - val_loss: 0.5169
Epoch 7/20
180/180 ━━━━━━━━━━━━━━━━━━━━ 20s 112ms/step - accuracy: 0.8312 - loss: 0.4667 - val_accuracy: 0.7937 - val_loss: 0.4965
Epoch 8/20
180/180 ━━━━━━━━━━━━━━━━━━━━ 20s 111ms/step - accura

In [22]:
save_results("amp_antibp2_25", accuracy_per_fold, auc_per_fold, all_classification_reports)
save_classification_report("amp_antibp2_25", avg_classification_report)

Results saved to reports/nosf/classification_results3_CNN.json
Classification report saved to reports/nosf/classification_reports3_cnn.json


In [23]:
accuracy_per_fold, auc_per_fold, all_classification_reports, avg_accuracy, std_accuracy, avg_auc, std_auc, avg_classification_report  = cross_validate_model(image_folder_csamp, labels_file_csamp, num_folds=10)


Training fold 1...
Epoch 1/20
23/23 ━━━━━━━━━━━━━━━━━━━━ 4s 125ms/step - accuracy: 0.5077 - loss: 1.1408 - val_accuracy: 0.4762 - val_loss: 0.8134
Epoch 2/20
23/23 ━━━━━━━━━━━━━━━━━━━━ 3s 114ms/step - accuracy: 0.4501 - loss: 0.8091 - val_accuracy: 0.6667 - val_loss: 0.7900
Epoch 3/20
23/23 ━━━━━━━━━━━━━━━━━━━━ 3s 112ms/step - accuracy: 0.5299 - loss: 0.7873 - val_accuracy: 0.7143 - val_loss: 0.7731
Epoch 4/20
23/23 ━━━━━━━━━━━━━━━━━━━━ 3s 114ms/step - accuracy: 0.6398 - loss: 0.7705 - val_accuracy: 0.7143 - val_loss: 0.7549
Epoch 5/20
23/23 ━━━━━━━━━━━━━━━━━━━━ 3s 111ms/step - accuracy: 0.6501 - loss: 0.7525 - val_accuracy: 0.6667 - val_loss: 0.7418
Epoch 6/20
23/23 ━━━━━━━━━━━━━━━━━━━━ 3s 111ms/step - accuracy: 0.6535 - loss: 0.7367 - val_accuracy: 0.6190 - val_loss: 0.7264
Epoch 7/20
23/23 ━━━━━━━━━━━━━━━━━━━━ 3s 110ms/step - accuracy: 0.5958 - loss: 0.7090 - val_accuracy: 0.8095 - val_loss: 0.6545
Epoch 8/20
23/23 ━━━━━━━━━━━━━━━━━━━━ 3s 111ms/step - accuracy: 0.7522 - loss: 0.628

In [24]:
save_results("amp_csamp_25", accuracy_per_fold, auc_per_fold, all_classification_reports)
save_classification_report("amp_csamp_25", avg_classification_report)

Results saved to reports/nosf/classification_results3_CNN.json
Classification report saved to reports/nosf/classification_reports3_cnn.json


In [25]:
accuracy_per_fold, auc_per_fold, all_classification_reports, avg_accuracy, std_accuracy, avg_auc, std_auc, avg_classification_report  = cross_validate_model(image_folder_hivddi, labels_file_hivddi, num_folds=10)


Training fold 1...
Epoch 1/20
56/56 ━━━━━━━━━━━━━━━━━━━━ 7s 112ms/step - accuracy: 0.5300 - loss: 1.0400 - val_accuracy: 0.5800 - val_loss: 0.7816
Epoch 2/20
56/56 ━━━━━━━━━━━━━━━━━━━━ 6s 108ms/step - accuracy: 0.4969 - loss: 0.7759 - val_accuracy: 0.5800 - val_loss: 0.7573
Epoch 3/20
56/56 ━━━━━━━━━━━━━━━━━━━━ 6s 108ms/step - accuracy: 0.4822 - loss: 0.7549 - val_accuracy: 0.4200 - val_loss: 0.7452
Epoch 4/20
56/56 ━━━━━━━━━━━━━━━━━━━━ 6s 109ms/step - accuracy: 0.5135 - loss: 0.7409 - val_accuracy: 0.7200 - val_loss: 0.7340
Epoch 5/20
56/56 ━━━━━━━━━━━━━━━━━━━━ 6s 107ms/step - accuracy: 0.5988 - loss: 0.7316 - val_accuracy: 0.5800 - val_loss: 0.7249
Epoch 6/20
56/56 ━━━━━━━━━━━━━━━━━━━━ 6s 110ms/step - accuracy: 0.4907 - loss: 0.7274 - val_accuracy: 0.5000 - val_loss: 0.7228
Epoch 7/20
56/56 ━━━━━━━━━━━━━━━━━━━━ 6s 110ms/step - accuracy: 0.5296 - loss: 0.7219 - val_accuracy: 0.5800 - val_loss: 0.7192
Epoch 8/20
56/56 ━━━━━━━━━━━━━━━━━━━━ 6s 109ms/step - accuracy: 0.5563 - loss: 0.717

In [26]:
save_results("hiv_ddi_25", accuracy_per_fold, auc_per_fold, all_classification_reports)
save_classification_report("hiv_ddi_25", avg_classification_report)

Results saved to reports/nosf/classification_results3_CNN.json
Classification report saved to reports/nosf/classification_reports3_cnn.json


In [27]:
accuracy_per_fold, auc_per_fold, all_classification_reports, avg_accuracy, std_accuracy, avg_auc, std_auc, avg_classification_report = cross_validate_model(image_folder_hivrtv, labels_file_hivrtv, num_folds=10)


Training fold 1...
Epoch 1/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 10s 129ms/step - accuracy: 0.4981 - loss: 1.0578 - val_accuracy: 0.5593 - val_loss: 0.7792
Epoch 2/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 8s 121ms/step - accuracy: 0.5906 - loss: 0.7704 - val_accuracy: 0.6271 - val_loss: 0.7521
Epoch 3/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 8s 121ms/step - accuracy: 0.6374 - loss: 0.7329 - val_accuracy: 0.7627 - val_loss: 0.7007
Epoch 4/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 8s 125ms/step - accuracy: 0.6690 - loss: 0.6806 - val_accuracy: 0.5763 - val_loss: 0.7223
Epoch 5/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 8s 125ms/step - accuracy: 0.7304 - loss: 0.6198 - val_accuracy: 0.6102 - val_loss: 0.6302
Epoch 6/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 8s 123ms/step - accuracy: 0.7374 - loss: 0.5602 - val_accuracy: 0.7797 - val_loss: 0.5838
Epoch 7/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 8s 123ms/step - accuracy: 0.7915 - loss: 0.5336 - val_accuracy: 0.7797 - val_loss: 0.5519
Epoch 8/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 8s 123ms/step - accuracy: 0.8168 - loss: 0.47

In [28]:
save_results("hiv_rtv_25", accuracy_per_fold, auc_per_fold, all_classification_reports)
save_classification_report("hiv_rtv_25", avg_classification_report)

Results saved to reports/nosf/classification_results3_CNN.json
Classification report saved to reports/nosf/classification_reports3_cnn.json


Files paths RES 50


In [9]:
image_folder_antiinflam = 'data/images/img_nosf/img50/aip_antiinflam'
labels_file_antiinflam = 'data/labels/aip_antiinflam.txt'

In [10]:
image_folder_antipb = 'data/images/img_nosf/img50/amp_antibp'
labels_file_antipb = 'data/labels/amp_antibp.txt'


In [11]:
image_folder_antipb2 = 'data/images/img_nosf/img50/amp_antibp2'
labels_file_antipb2 = 'data/labels/amp_antibp2.txt'

In [12]:
image_folder_csamp = 'data/images/img_nosf/img50/amp_csamp'
labels_file_csamp = 'data/labels/amp_csamp.txt'

In [13]:
image_folder_hivddi = 'data/images/img_nosf/img50/hiv_ddi'
labels_file_hivddi = 'data/labels/hiv_ddi.txt'

In [14]:
image_folder_hivrtv = 'data/images/img_nosf/img50/hiv_rtv'
labels_file_hivrtv = 'data/labels/hiv_rtv.txt'

TRAINING RES 50

In [15]:
accuracy_per_fold, auc_per_fold, all_classification_reports, avg_accuracy, std_accuracy, avg_auc, std_auc, avg_classification_report  = cross_validate_model(image_folder_antiinflam, labels_file_antiinflam, num_folds=10)


Training fold 1...
Epoch 1/20
192/192 ━━━━━━━━━━━━━━━━━━━━ 24s 117ms/step - accuracy: 0.5678 - loss: 0.9236 - val_accuracy: 0.6176 - val_loss: 0.7126
Epoch 2/20
192/192 ━━━━━━━━━━━━━━━━━━━━ 22s 115ms/step - accuracy: 0.5975 - loss: 0.7225 - val_accuracy: 0.6176 - val_loss: 0.6945
Epoch 3/20
192/192 ━━━━━━━━━━━━━━━━━━━━ 22s 116ms/step - accuracy: 0.5637 - loss: 0.7008 - val_accuracy: 0.6118 - val_loss: 0.6572
Epoch 4/20
192/192 ━━━━━━━━━━━━━━━━━━━━ 22s 117ms/step - accuracy: 0.6242 - loss: 0.6557 - val_accuracy: 0.6529 - val_loss: 0.6342
Epoch 5/20
192/192 ━━━━━━━━━━━━━━━━━━━━ 23s 118ms/step - accuracy: 0.7000 - loss: 0.6022 - val_accuracy: 0.6588 - val_loss: 0.6152
Epoch 6/20
192/192 ━━━━━━━━━━━━━━━━━━━━ 22s 115ms/step - accuracy: 0.7127 - loss: 0.5735 - val_accuracy: 0.6235 - val_loss: 0.6370
Epoch 7/20
192/192 ━━━━━━━━━━━━━━━━━━━━ 22s 114ms/step - accuracy: 0.7464 - loss: 0.5438 - val_accuracy: 0.7118 - val_loss: 0.5982
Epoch 8/20
192/192 ━━━━━━━━━━━━━━━━━━━━ 22s 114ms/step - accura

In [16]:
save_results("aip_antiinflam_50", accuracy_per_fold, auc_per_fold, all_classification_reports)
save_classification_report("aip_antiinflam_50", avg_classification_report)

Results saved to reports/nosf/classification_results3_CNN.json
Classification report saved to reports/nosf/classification_reports3_cnn.json


In [17]:
accuracy_per_fold, auc_per_fold, all_classification_reports, avg_accuracy, std_accuracy, avg_auc, std_auc, avg_classification_report = cross_validate_model(image_folder_antipb, labels_file_antipb, num_folds=10) 


Training fold 1...
Epoch 1/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 10s 111ms/step - accuracy: 0.5023 - loss: 0.9843 - val_accuracy: 0.4783 - val_loss: 0.7719
Epoch 2/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 8s 107ms/step - accuracy: 0.5548 - loss: 0.7672 - val_accuracy: 0.5362 - val_loss: 0.7168
Epoch 3/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 8s 107ms/step - accuracy: 0.6560 - loss: 0.7180 - val_accuracy: 0.4783 - val_loss: 0.6983
Epoch 4/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 8s 108ms/step - accuracy: 0.6808 - loss: 0.6418 - val_accuracy: 0.8551 - val_loss: 0.4709
Epoch 5/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 8s 107ms/step - accuracy: 0.8091 - loss: 0.5165 - val_accuracy: 0.9130 - val_loss: 0.4116
Epoch 6/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 8s 108ms/step - accuracy: 0.8099 - loss: 0.4930 - val_accuracy: 0.9275 - val_loss: 0.3838
Epoch 7/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 8s 108ms/step - accuracy: 0.8725 - loss: 0.4174 - val_accuracy: 0.8696 - val_loss: 0.4107
Epoch 8/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 9s 109ms/step - accuracy: 0.8471 - loss: 0.42

In [18]:
save_results("amp_antibp_50", accuracy_per_fold, auc_per_fold, all_classification_reports)
save_classification_report("amp_antibp_50", avg_classification_report)

Results saved to reports/nosf/classification_results3_CNN.json
Classification report saved to reports/nosf/classification_reports3_cnn.json


In [19]:
accuracy_per_fold, auc_per_fold, all_classification_reports, avg_accuracy, std_accuracy, avg_auc, std_auc, avg_classification_report  = cross_validate_model(image_folder_antipb2, labels_file_antipb2, num_folds=10)


Training fold 1...
Epoch 1/20
180/180 ━━━━━━━━━━━━━━━━━━━━ 21s 108ms/step - accuracy: 0.5232 - loss: 0.8943 - val_accuracy: 0.5688 - val_loss: 0.7136
Epoch 2/20
180/180 ━━━━━━━━━━━━━━━━━━━━ 19s 107ms/step - accuracy: 0.6639 - loss: 0.6877 - val_accuracy: 0.7812 - val_loss: 0.5700
Epoch 3/20
180/180 ━━━━━━━━━━━━━━━━━━━━ 19s 107ms/step - accuracy: 0.7450 - loss: 0.5677 - val_accuracy: 0.7500 - val_loss: 0.5366
Epoch 4/20
180/180 ━━━━━━━━━━━━━━━━━━━━ 19s 107ms/step - accuracy: 0.7537 - loss: 0.5532 - val_accuracy: 0.7625 - val_loss: 0.5436
Epoch 5/20
180/180 ━━━━━━━━━━━━━━━━━━━━ 19s 108ms/step - accuracy: 0.7651 - loss: 0.5439 - val_accuracy: 0.8000 - val_loss: 0.4905
Epoch 6/20
180/180 ━━━━━━━━━━━━━━━━━━━━ 19s 108ms/step - accuracy: 0.7881 - loss: 0.4999 - val_accuracy: 0.7937 - val_loss: 0.4833
Epoch 7/20
180/180 ━━━━━━━━━━━━━━━━━━━━ 19s 108ms/step - accuracy: 0.8077 - loss: 0.4819 - val_accuracy: 0.7812 - val_loss: 0.4927
Epoch 8/20
180/180 ━━━━━━━━━━━━━━━━━━━━ 20s 108ms/step - accura

In [20]:
save_results("amp_antibp2_50", accuracy_per_fold, auc_per_fold, all_classification_reports)
save_classification_report("amp_antibp2_50", avg_classification_report)

Results saved to reports/nosf/classification_results3_CNN.json
Classification report saved to reports/nosf/classification_reports3_cnn.json


In [21]:
accuracy_per_fold, auc_per_fold, all_classification_reports, avg_accuracy, std_accuracy, avg_auc, std_auc, avg_classification_report  = cross_validate_model(image_folder_csamp, labels_file_csamp, num_folds=10)


Training fold 1...
Epoch 1/20
23/23 ━━━━━━━━━━━━━━━━━━━━ 4s 121ms/step - accuracy: 0.4794 - loss: 1.2337 - val_accuracy: 0.4762 - val_loss: 0.8097
Epoch 2/20
23/23 ━━━━━━━━━━━━━━━━━━━━ 3s 109ms/step - accuracy: 0.5266 - loss: 0.8055 - val_accuracy: 0.5238 - val_loss: 0.7888
Epoch 3/20
23/23 ━━━━━━━━━━━━━━━━━━━━ 3s 107ms/step - accuracy: 0.5004 - loss: 0.7858 - val_accuracy: 0.5238 - val_loss: 0.7719
Epoch 4/20
23/23 ━━━━━━━━━━━━━━━━━━━━ 3s 111ms/step - accuracy: 0.4760 - loss: 0.7728 - val_accuracy: 0.6667 - val_loss: 0.7569
Epoch 5/20
23/23 ━━━━━━━━━━━━━━━━━━━━ 3s 109ms/step - accuracy: 0.5668 - loss: 0.7555 - val_accuracy: 0.8095 - val_loss: 0.7264
Epoch 6/20
23/23 ━━━━━━━━━━━━━━━━━━━━ 3s 109ms/step - accuracy: 0.5970 - loss: 0.7476 - val_accuracy: 0.7143 - val_loss: 0.7002
Epoch 7/20
23/23 ━━━━━━━━━━━━━━━━━━━━ 3s 108ms/step - accuracy: 0.5715 - loss: 0.7210 - val_accuracy: 0.6667 - val_loss: 0.6664
Epoch 8/20
23/23 ━━━━━━━━━━━━━━━━━━━━ 2s 107ms/step - accuracy: 0.6305 - loss: 0.698

In [22]:
save_results("amp_csamp_50", accuracy_per_fold, auc_per_fold, all_classification_reports)
save_classification_report("amp_csamp_50", avg_classification_report)

Results saved to reports/nosf/classification_results3_CNN.json
Classification report saved to reports/nosf/classification_reports3_cnn.json


In [23]:
accuracy_per_fold, auc_per_fold, all_classification_reports, avg_accuracy, std_accuracy, avg_auc, std_auc, avg_classification_report  = cross_validate_model(image_folder_hivddi, labels_file_hivddi, num_folds=10)


Training fold 1...
Epoch 1/20
56/56 ━━━━━━━━━━━━━━━━━━━━ 7s 111ms/step - accuracy: 0.5037 - loss: 1.1590 - val_accuracy: 0.4200 - val_loss: 0.7973
Epoch 2/20
56/56 ━━━━━━━━━━━━━━━━━━━━ 6s 108ms/step - accuracy: 0.4811 - loss: 0.7893 - val_accuracy: 0.5800 - val_loss: 0.7697
Epoch 3/20
56/56 ━━━━━━━━━━━━━━━━━━━━ 6s 108ms/step - accuracy: 0.4490 - loss: 0.7663 - val_accuracy: 0.5800 - val_loss: 0.7529
Epoch 4/20
56/56 ━━━━━━━━━━━━━━━━━━━━ 6s 108ms/step - accuracy: 0.5332 - loss: 0.7504 - val_accuracy: 0.5800 - val_loss: 0.7326
Epoch 5/20
56/56 ━━━━━━━━━━━━━━━━━━━━ 6s 108ms/step - accuracy: 0.5038 - loss: 0.7428 - val_accuracy: 0.4200 - val_loss: 0.7358
Epoch 6/20
56/56 ━━━━━━━━━━━━━━━━━━━━ 6s 108ms/step - accuracy: 0.4832 - loss: 0.7360 - val_accuracy: 0.4200 - val_loss: 0.7306
Epoch 7/20
56/56 ━━━━━━━━━━━━━━━━━━━━ 6s 108ms/step - accuracy: 0.5223 - loss: 0.7282 - val_accuracy: 0.5800 - val_loss: 0.7129
Epoch 8/20
56/56 ━━━━━━━━━━━━━━━━━━━━ 6s 109ms/step - accuracy: 0.5909 - loss: 0.721

In [24]:
save_results("hiv_ddi_50", accuracy_per_fold, auc_per_fold, all_classification_reports)
save_classification_report("hiv_ddi_50", avg_classification_report)

Results saved to reports/nosf/classification_results3_CNN.json
Classification report saved to reports/nosf/classification_reports3_cnn.json


In [25]:
accuracy_per_fold, auc_per_fold, all_classification_reports, avg_accuracy, std_accuracy, avg_auc, std_auc, avg_classification_report = cross_validate_model(image_folder_hivrtv, labels_file_hivrtv, num_folds=10 )


Training fold 1...
Epoch 1/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 9s 110ms/step - accuracy: 0.4807 - loss: 1.4071 - val_accuracy: 0.5593 - val_loss: 0.7802
Epoch 2/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 7s 107ms/step - accuracy: 0.4301 - loss: 0.7755 - val_accuracy: 0.5593 - val_loss: 0.7578
Epoch 3/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 7s 108ms/step - accuracy: 0.5372 - loss: 0.7546 - val_accuracy: 0.5593 - val_loss: 0.7401
Epoch 4/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 7s 108ms/step - accuracy: 0.5240 - loss: 0.7428 - val_accuracy: 0.5593 - val_loss: 0.7366
Epoch 5/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 7s 108ms/step - accuracy: 0.4939 - loss: 0.7353 - val_accuracy: 0.5593 - val_loss: 0.7295
Epoch 6/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 7s 107ms/step - accuracy: 0.5110 - loss: 0.7293 - val_accuracy: 0.5593 - val_loss: 0.7239
Epoch 7/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 7s 108ms/step - accuracy: 0.5386 - loss: 0.7236 - val_accuracy: 0.5593 - val_loss: 0.7205
Epoch 8/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 7s 107ms/step - accuracy: 0.5361 - loss: 0.720

In [26]:
save_results("hiv_rtv_50", accuracy_per_fold, auc_per_fold, all_classification_reports)
save_classification_report("hiv_rtv_50", avg_classification_report)

Results saved to reports/nosf/classification_results3_CNN.json
Classification report saved to reports/nosf/classification_reports3_cnn.json


File paths RES 75

In [10]:
image_folder_antiinflam = 'data/images/img_nosf/img75/aip_antiinflam' 
labels_file_antiinflam = 'data/labels/aip_antiinflam.txt'

In [11]:
image_folder_antipb = 'data/images/img_nosf/img75/amp_antibp'
labels_file_antipb = 'data/labels/amp_antibp.txt'


In [12]:
image_folder_antipb2 = 'data/images/img_nosf/img75/amp_antibp2'
labels_file_antipb2 = 'data/labels/amp_antibp2.txt'

In [13]:
image_folder_csamp = 'data/images/img_nosf/img75/amp_csamp'
labels_file_csamp = 'data/labels/amp_csamp.txt'

In [14]:
image_folder_hivddi = 'data/images/img_nosf/img75/hiv_ddi'
labels_file_hivddi = 'data/labels/hiv_ddi.txt'

In [15]:
image_folder_hivrtv = 'data/images/img_nosf/img75/hiv_rtv'
labels_file_hivrtv = 'data/labels/hiv_rtv.txt'

TRAINING RES 75

In [33]:
accuracy_per_fold, auc_per_fold, all_classification_reports, avg_accuracy, std_accuracy, avg_auc, std_auc, avg_classification_report  = cross_validate_model(image_folder_antiinflam, labels_file_antiinflam, num_folds=10)


Training fold 1...
Epoch 1/20
192/192 ━━━━━━━━━━━━━━━━━━━━ 24s 115ms/step - accuracy: 0.5369 - loss: 0.9237 - val_accuracy: 0.6176 - val_loss: 0.7239
Epoch 2/20
192/192 ━━━━━━━━━━━━━━━━━━━━ 21s 110ms/step - accuracy: 0.5829 - loss: 0.7353 - val_accuracy: 0.6176 - val_loss: 0.7320
Epoch 3/20
192/192 ━━━━━━━━━━━━━━━━━━━━ 21s 112ms/step - accuracy: 0.6082 - loss: 0.7355 - val_accuracy: 0.6176 - val_loss: 0.7041
Epoch 4/20
192/192 ━━━━━━━━━━━━━━━━━━━━ 21s 112ms/step - accuracy: 0.6051 - loss: 0.6988 - val_accuracy: 0.6176 - val_loss: 0.6872
Epoch 5/20
192/192 ━━━━━━━━━━━━━━━━━━━━ 22s 113ms/step - accuracy: 0.5866 - loss: 0.7015 - val_accuracy: 0.6176 - val_loss: 0.6851
Epoch 6/20
192/192 ━━━━━━━━━━━━━━━━━━━━ 21s 111ms/step - accuracy: 0.5880 - loss: 0.6976 - val_accuracy: 0.6176 - val_loss: 0.6873
Epoch 7/20
192/192 ━━━━━━━━━━━━━━━━━━━━ 21s 111ms/step - accuracy: 0.5923 - loss: 0.6946 - val_accuracy: 0.6176 - val_loss: 0.6790
Epoch 8/20
192/192 ━━━━━━━━━━━━━━━━━━━━ 21s 111ms/step - accura

In [34]:
save_results("aip_antiinflam_75", accuracy_per_fold, auc_per_fold, all_classification_reports)
save_classification_report("aip_antiinflam_75", avg_classification_report)

Results saved to reports/nosf/classification_results3_CNN.json
Classification report saved to reports/nosf/classification_reports3_cnn.json


In [16]:
accuracy_per_fold, auc_per_fold, all_classification_reports, avg_accuracy, std_accuracy, avg_auc, std_auc, avg_classification_report= cross_validate_model(image_folder_antipb, labels_file_antipb, num_folds=10)


Training fold 1...
Epoch 1/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 10s 107ms/step - accuracy: 0.4654 - loss: 1.0807 - val_accuracy: 0.4783 - val_loss: 0.7819
Epoch 2/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 9s 111ms/step - accuracy: 0.5289 - loss: 0.7743 - val_accuracy: 0.6667 - val_loss: 0.7340
Epoch 3/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 8s 107ms/step - accuracy: 0.5635 - loss: 0.7414 - val_accuracy: 0.7246 - val_loss: 0.6419
Epoch 4/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 8s 107ms/step - accuracy: 0.6806 - loss: 0.6479 - val_accuracy: 0.8406 - val_loss: 0.5121
Epoch 5/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 8s 108ms/step - accuracy: 0.7800 - loss: 0.5517 - val_accuracy: 0.8696 - val_loss: 0.4749
Epoch 6/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 8s 108ms/step - accuracy: 0.8124 - loss: 0.5160 - val_accuracy: 0.9130 - val_loss: 0.3996
Epoch 7/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 8s 107ms/step - accuracy: 0.8091 - loss: 0.4773 - val_accuracy: 0.8986 - val_loss: 0.4062
Epoch 8/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 9s 115ms/step - accuracy: 0.8193 - loss: 0.49

In [17]:
save_results("amp_antibp_75", accuracy_per_fold, auc_per_fold, all_classification_reports)
save_classification_report("amp_antibp_75", avg_classification_report)

Results saved to reports/nosf/classification_results3_CNN.json
Classification report saved to reports/nosf/classification_reports3_cnn.json


In [18]:
accuracy_per_fold, auc_per_fold, all_classification_reports, avg_accuracy, std_accuracy, avg_auc, std_auc, avg_classification_report  = cross_validate_model(image_folder_antipb2, labels_file_antipb2, num_folds=10)


Training fold 1...
Epoch 1/20
180/180 ━━━━━━━━━━━━━━━━━━━━ 20s 102ms/step - accuracy: 0.4945 - loss: 0.9619 - val_accuracy: 0.6438 - val_loss: 0.7546
Epoch 2/20
180/180 ━━━━━━━━━━━━━━━━━━━━ 18s 101ms/step - accuracy: 0.5244 - loss: 0.7494 - val_accuracy: 0.4625 - val_loss: 0.7321
Epoch 3/20
180/180 ━━━━━━━━━━━━━━━━━━━━ 18s 102ms/step - accuracy: 0.5018 - loss: 0.7271 - val_accuracy: 0.4625 - val_loss: 0.7180
Epoch 4/20
180/180 ━━━━━━━━━━━━━━━━━━━━ 18s 101ms/step - accuracy: 0.5406 - loss: 0.7178 - val_accuracy: 0.4625 - val_loss: 0.7122
Epoch 5/20
180/180 ━━━━━━━━━━━━━━━━━━━━ 18s 102ms/step - accuracy: 0.5208 - loss: 0.7101 - val_accuracy: 0.4625 - val_loss: 0.7075
Epoch 6/20
180/180 ━━━━━━━━━━━━━━━━━━━━ 18s 102ms/step - accuracy: 0.5025 - loss: 0.7064 - val_accuracy: 0.4625 - val_loss: 0.7046
Epoch 7/20
180/180 ━━━━━━━━━━━━━━━━━━━━ 18s 102ms/step - accuracy: 0.5162 - loss: 0.7035 - val_accuracy: 0.4625 - val_loss: 0.7030
Epoch 8/20
180/180 ━━━━━━━━━━━━━━━━━━━━ 18s 102ms/step - accura

In [19]:
save_results("amp_antibp2_75", accuracy_per_fold, auc_per_fold, all_classification_reports)
save_classification_report("amp_antibp2_75", avg_classification_report)

Results saved to reports/nosf/classification_results3_CNN.json
Classification report saved to reports/nosf/classification_reports3_cnn.json


In [20]:
accuracy_per_fold, auc_per_fold, all_classification_reports, avg_accuracy, std_accuracy, avg_auc, std_auc, avg_classification_report = cross_validate_model(image_folder_csamp, labels_file_csamp, num_folds=10)


Training fold 1...
Epoch 1/20
23/23 ━━━━━━━━━━━━━━━━━━━━ 4s 118ms/step - accuracy: 0.5218 - loss: 1.4626 - val_accuracy: 0.5238 - val_loss: 0.8267
Epoch 2/20
23/23 ━━━━━━━━━━━━━━━━━━━━ 2s 107ms/step - accuracy: 0.5892 - loss: 0.8167 - val_accuracy: 0.4762 - val_loss: 0.8093
Epoch 3/20
23/23 ━━━━━━━━━━━━━━━━━━━━ 3s 109ms/step - accuracy: 0.5342 - loss: 0.7968 - val_accuracy: 0.5238 - val_loss: 0.7882
Epoch 4/20
23/23 ━━━━━━━━━━━━━━━━━━━━ 2s 107ms/step - accuracy: 0.5128 - loss: 0.7873 - val_accuracy: 0.6667 - val_loss: 0.7755
Epoch 5/20
23/23 ━━━━━━━━━━━━━━━━━━━━ 3s 108ms/step - accuracy: 0.5216 - loss: 0.7773 - val_accuracy: 0.7143 - val_loss: 0.7621
Epoch 6/20
23/23 ━━━━━━━━━━━━━━━━━━━━ 3s 110ms/step - accuracy: 0.5668 - loss: 0.7627 - val_accuracy: 0.7143 - val_loss: 0.7387
Epoch 7/20
23/23 ━━━━━━━━━━━━━━━━━━━━ 3s 108ms/step - accuracy: 0.5690 - loss: 0.7515 - val_accuracy: 0.6667 - val_loss: 0.7144
Epoch 8/20
23/23 ━━━━━━━━━━━━━━━━━━━━ 3s 108ms/step - accuracy: 0.6154 - loss: 0.733

In [21]:
save_results("amp_csamp_75", accuracy_per_fold, auc_per_fold, all_classification_reports)
save_classification_report("amp_csamp_75", avg_classification_report)

Results saved to reports/nosf/classification_results3_CNN.json
Classification report saved to reports/nosf/classification_reports3_cnn.json


In [22]:
accuracy_per_fold, auc_per_fold, all_classification_reports, avg_accuracy, std_accuracy, avg_auc, std_auc, avg_classification_report  = cross_validate_model(image_folder_hivddi, labels_file_hivddi, num_folds=10)


Training fold 1...
Epoch 1/20
56/56 ━━━━━━━━━━━━━━━━━━━━ 7s 111ms/step - accuracy: 0.5035 - loss: 1.0566 - val_accuracy: 0.4200 - val_loss: 0.7825
Epoch 2/20
56/56 ━━━━━━━━━━━━━━━━━━━━ 6s 107ms/step - accuracy: 0.5372 - loss: 0.7729 - val_accuracy: 0.4200 - val_loss: 0.7580
Epoch 3/20
56/56 ━━━━━━━━━━━━━━━━━━━━ 6s 107ms/step - accuracy: 0.4876 - loss: 0.7517 - val_accuracy: 0.4200 - val_loss: 0.7425
Epoch 4/20
56/56 ━━━━━━━━━━━━━━━━━━━━ 6s 107ms/step - accuracy: 0.5249 - loss: 0.7387 - val_accuracy: 0.4200 - val_loss: 0.7325
Epoch 5/20
56/56 ━━━━━━━━━━━━━━━━━━━━ 6s 106ms/step - accuracy: 0.4893 - loss: 0.7304 - val_accuracy: 0.4200 - val_loss: 0.7360
Epoch 6/20
56/56 ━━━━━━━━━━━━━━━━━━━━ 6s 109ms/step - accuracy: 0.5149 - loss: 0.7293 - val_accuracy: 0.4200 - val_loss: 0.7210
Epoch 7/20
56/56 ━━━━━━━━━━━━━━━━━━━━ 6s 108ms/step - accuracy: 0.4451 - loss: 0.7212 - val_accuracy: 0.4200 - val_loss: 0.7181
Epoch 8/20
56/56 ━━━━━━━━━━━━━━━━━━━━ 6s 108ms/step - accuracy: 0.4027 - loss: 0.717

In [23]:
save_results("hiv_ddi_75", accuracy_per_fold, auc_per_fold, all_classification_reports)
save_classification_report("hiv_ddi_75", avg_classification_report)

Results saved to reports/nosf/classification_results3_CNN.json
Classification report saved to reports/nosf/classification_reports3_cnn.json


In [24]:
accuracy_per_fold, auc_per_fold, all_classification_reports, avg_accuracy, std_accuracy, avg_auc, std_auc, avg_classification_report= cross_validate_model(image_folder_hivrtv, labels_file_hivrtv, num_folds=10)


Training fold 1...
Epoch 1/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 9s 109ms/step - accuracy: 0.5189 - loss: 1.0713 - val_accuracy: 0.4407 - val_loss: 0.7708
Epoch 2/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 7s 106ms/step - accuracy: 0.5357 - loss: 0.7678 - val_accuracy: 0.4407 - val_loss: 0.7467
Epoch 3/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 7s 105ms/step - accuracy: 0.5333 - loss: 0.7429 - val_accuracy: 0.5593 - val_loss: 0.7301
Epoch 4/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 7s 107ms/step - accuracy: 0.5272 - loss: 0.7345 - val_accuracy: 0.4407 - val_loss: 0.7278
Epoch 5/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 7s 108ms/step - accuracy: 0.4373 - loss: 0.7270 - val_accuracy: 0.5593 - val_loss: 0.7225
Epoch 6/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 7s 108ms/step - accuracy: 0.4802 - loss: 0.7218 - val_accuracy: 0.5593 - val_loss: 0.7186
Epoch 7/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 7s 108ms/step - accuracy: 0.5084 - loss: 0.7180 - val_accuracy: 0.5593 - val_loss: 0.7155
Epoch 8/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 7s 108ms/step - accuracy: 0.5304 - loss: 0.715

In [25]:
save_results("hiv_rtv_75", accuracy_per_fold, auc_per_fold, all_classification_reports)
save_classification_report("hiv_rtv_75", avg_classification_report)

Results saved to reports/nosf/classification_results3_CNN.json
Classification report saved to reports/nosf/classification_reports3_cnn.json


File paths RES 100

In [26]:
image_folder_antiinflam = 'data/images/img_nosf/img100/aip_antiinflam'
labels_file_antiinflam = 'data/labels/aip_antiinflam.txt'

In [27]:
image_folder_antipb = 'data/images/img_nosf/img100/amp_antibp'
labels_file_antipb = 'data/labels/amp_antibp.txt'


In [28]:
image_folder_antipb2 = 'data/images/img_nosf/img100/amp_antibp2'
labels_file_antipb2 = 'data/labels/amp_antibp2.txt'

In [29]:
image_folder_csamp = 'data/images/img_nosf/img100/amp_csamp'
labels_file_csamp = 'data/labels/amp_csamp.txt'

In [30]:
image_folder_hivddi = 'data/images/img_nosf/img100/hiv_ddi'
labels_file_hivddi = 'data/labels/hiv_ddi.txt'

In [31]:
image_folder_hivrtv = 'data/images/img_nosf/img100/hiv_rtv'
labels_file_hivrtv = 'data/labels/hiv_rtv.txt'

TRAINING RES 100

In [32]:
accuracy_per_fold, auc_per_fold, all_classification_reports, avg_accuracy, std_accuracy, avg_auc, std_auc, avg_classification_report  = cross_validate_model(image_folder_antiinflam, labels_file_antiinflam, num_folds=10)


Training fold 1...
Epoch 1/20
192/192 ━━━━━━━━━━━━━━━━━━━━ 22s 107ms/step - accuracy: 0.5215 - loss: 1.0732 - val_accuracy: 0.6176 - val_loss: 0.7359
Epoch 2/20
192/192 ━━━━━━━━━━━━━━━━━━━━ 21s 107ms/step - accuracy: 0.5644 - loss: 0.7542 - val_accuracy: 0.6176 - val_loss: 0.7124
Epoch 3/20
192/192 ━━━━━━━━━━━━━━━━━━━━ 20s 106ms/step - accuracy: 0.5944 - loss: 0.7221 - val_accuracy: 0.6176 - val_loss: 0.7038
Epoch 4/20
192/192 ━━━━━━━━━━━━━━━━━━━━ 21s 107ms/step - accuracy: 0.5986 - loss: 0.7070 - val_accuracy: 0.6176 - val_loss: 0.6952
Epoch 5/20
192/192 ━━━━━━━━━━━━━━━━━━━━ 21s 107ms/step - accuracy: 0.6197 - loss: 0.6961 - val_accuracy: 0.6176 - val_loss: 0.6924
Epoch 6/20
192/192 ━━━━━━━━━━━━━━━━━━━━ 21s 107ms/step - accuracy: 0.5989 - loss: 0.6975 - val_accuracy: 0.6176 - val_loss: 0.6892
Epoch 7/20
192/192 ━━━━━━━━━━━━━━━━━━━━ 21s 108ms/step - accuracy: 0.5836 - loss: 0.7010 - val_accuracy: 0.6176 - val_loss: 0.6828
Epoch 8/20
192/192 ━━━━━━━━━━━━━━━━━━━━ 21s 108ms/step - accura

In [33]:
save_results("aip_antiinflam_100", accuracy_per_fold, auc_per_fold, all_classification_reports)
save_classification_report("aip_antiinflam_100", avg_classification_report)

Results saved to reports/nosf/classification_results3_CNN.json
Classification report saved to reports/nosf/classification_reports3_cnn.json


In [34]:
accuracy_per_fold, auc_per_fold, all_classification_reports, avg_accuracy, std_accuracy, avg_auc, std_auc, avg_classification_report = cross_validate_model(image_folder_antipb, labels_file_antipb, num_folds=10)


Training fold 1...
Epoch 1/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 10s 110ms/step - accuracy: 0.5020 - loss: 1.1413 - val_accuracy: 0.5217 - val_loss: 0.7759
Epoch 2/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 8s 106ms/step - accuracy: 0.5101 - loss: 0.7684 - val_accuracy: 0.5362 - val_loss: 0.7508
Epoch 3/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 8s 107ms/step - accuracy: 0.4689 - loss: 0.7476 - val_accuracy: 0.5217 - val_loss: 0.7418
Epoch 4/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 8s 107ms/step - accuracy: 0.4939 - loss: 0.7443 - val_accuracy: 0.8551 - val_loss: 0.7261
Epoch 5/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 9s 109ms/step - accuracy: 0.5895 - loss: 0.7257 - val_accuracy: 0.6087 - val_loss: 0.7205
Epoch 6/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 8s 108ms/step - accuracy: 0.5269 - loss: 0.7207 - val_accuracy: 0.4783 - val_loss: 0.7079
Epoch 7/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 8s 108ms/step - accuracy: 0.6051 - loss: 0.7036 - val_accuracy: 0.6957 - val_loss: 0.6806
Epoch 8/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 8s 108ms/step - accuracy: 0.6868 - loss: 0.68

In [35]:
save_results("amp_antibp_100", accuracy_per_fold, auc_per_fold, all_classification_reports)
save_classification_report("amp_antibp_100", avg_classification_report)

Results saved to reports/nosf/classification_results3_CNN.json
Classification report saved to reports/nosf/classification_reports3_cnn.json


In [36]:
accuracy_per_fold, auc_per_fold, all_classification_reports, avg_accuracy, std_accuracy, avg_auc, std_auc, avg_classification_report  = cross_validate_model(image_folder_antipb2, labels_file_antipb2, num_folds=10)


Training fold 1...
Epoch 1/20
180/180 ━━━━━━━━━━━━━━━━━━━━ 21s 108ms/step - accuracy: 0.4743 - loss: 0.8288 - val_accuracy: 0.4625 - val_loss: 0.7268
Epoch 2/20
180/180 ━━━━━━━━━━━━━━━━━━━━ 19s 107ms/step - accuracy: 0.5042 - loss: 0.7200 - val_accuracy: 0.4625 - val_loss: 0.7104
Epoch 3/20
180/180 ━━━━━━━━━━━━━━━━━━━━ 19s 107ms/step - accuracy: 0.5108 - loss: 0.7084 - val_accuracy: 0.4625 - val_loss: 0.7050
Epoch 4/20
180/180 ━━━━━━━━━━━━━━━━━━━━ 19s 108ms/step - accuracy: 0.5104 - loss: 0.7044 - val_accuracy: 0.4625 - val_loss: 0.7023
Epoch 5/20
180/180 ━━━━━━━━━━━━━━━━━━━━ 19s 108ms/step - accuracy: 0.4830 - loss: 0.7012 - val_accuracy: 0.4625 - val_loss: 0.7003
Epoch 6/20
180/180 ━━━━━━━━━━━━━━━━━━━━ 19s 108ms/step - accuracy: 0.5104 - loss: 0.6994 - val_accuracy: 0.4625 - val_loss: 0.6991
Epoch 7/20
180/180 ━━━━━━━━━━━━━━━━━━━━ 19s 108ms/step - accuracy: 0.5198 - loss: 0.6979 - val_accuracy: 0.4625 - val_loss: 0.6983
Epoch 8/20
180/180 ━━━━━━━━━━━━━━━━━━━━ 20s 108ms/step - accura

In [37]:
save_results("amp_antibp2_100", accuracy_per_fold, auc_per_fold, all_classification_reports)
save_classification_report("amp_antibp2_100", avg_classification_report)

Results saved to reports/nosf/classification_results3_CNN.json
Classification report saved to reports/nosf/classification_reports3_cnn.json


In [38]:
accuracy_per_fold, auc_per_fold, all_classification_reports, avg_accuracy, std_accuracy, avg_auc, std_auc, avg_classification_report = cross_validate_model(image_folder_csamp, labels_file_csamp, num_folds=10)


Training fold 1...
Epoch 1/20
23/23 ━━━━━━━━━━━━━━━━━━━━ 4s 117ms/step - accuracy: 0.5013 - loss: 1.1169 - val_accuracy: 0.5714 - val_loss: 0.7934
Epoch 2/20
23/23 ━━━━━━━━━━━━━━━━━━━━ 2s 108ms/step - accuracy: 0.5016 - loss: 0.7887 - val_accuracy: 0.5238 - val_loss: 0.7748
Epoch 3/20
23/23 ━━━━━━━━━━━━━━━━━━━━ 3s 109ms/step - accuracy: 0.5309 - loss: 0.7713 - val_accuracy: 0.4762 - val_loss: 0.7619
Epoch 4/20
23/23 ━━━━━━━━━━━━━━━━━━━━ 2s 106ms/step - accuracy: 0.4652 - loss: 0.7596 - val_accuracy: 0.4762 - val_loss: 0.7527
Epoch 5/20
23/23 ━━━━━━━━━━━━━━━━━━━━ 2s 107ms/step - accuracy: 0.4544 - loss: 0.7514 - val_accuracy: 0.4762 - val_loss: 0.7458
Epoch 6/20
23/23 ━━━━━━━━━━━━━━━━━━━━ 3s 108ms/step - accuracy: 0.5027 - loss: 0.7447 - val_accuracy: 0.5238 - val_loss: 0.7401
Epoch 7/20
23/23 ━━━━━━━━━━━━━━━━━━━━ 2s 107ms/step - accuracy: 0.5340 - loss: 0.7389 - val_accuracy: 0.4762 - val_loss: 0.7356
Epoch 8/20
23/23 ━━━━━━━━━━━━━━━━━━━━ 2s 107ms/step - accuracy: 0.5226 - loss: 0.734

In [39]:
save_results("amp_csamp_100", accuracy_per_fold, auc_per_fold, all_classification_reports)
save_classification_report("amp_csamp_100", avg_classification_report)

Results saved to reports/nosf/classification_results3_CNN.json
Classification report saved to reports/nosf/classification_reports3_cnn.json


In [40]:
accuracy_per_fold, auc_per_fold, all_classification_reports, avg_accuracy, std_accuracy, avg_auc, std_auc, avg_classification_report  = cross_validate_model(image_folder_hivddi, labels_file_hivddi, num_folds=10)


Training fold 1...
Epoch 1/20
56/56 ━━━━━━━━━━━━━━━━━━━━ 7s 115ms/step - accuracy: 0.5276 - loss: 0.9636 - val_accuracy: 0.4200 - val_loss: 0.7720
Epoch 2/20
56/56 ━━━━━━━━━━━━━━━━━━━━ 6s 111ms/step - accuracy: 0.5051 - loss: 0.7635 - val_accuracy: 0.4200 - val_loss: 0.7481
Epoch 3/20
56/56 ━━━━━━━━━━━━━━━━━━━━ 6s 111ms/step - accuracy: 0.5453 - loss: 0.7430 - val_accuracy: 0.5800 - val_loss: 0.7301
Epoch 4/20
56/56 ━━━━━━━━━━━━━━━━━━━━ 6s 112ms/step - accuracy: 0.4898 - loss: 0.7336 - val_accuracy: 0.4200 - val_loss: 0.7298
Epoch 5/20
56/56 ━━━━━━━━━━━━━━━━━━━━ 6s 113ms/step - accuracy: 0.5189 - loss: 0.7260 - val_accuracy: 0.4200 - val_loss: 0.7229
Epoch 6/20
56/56 ━━━━━━━━━━━━━━━━━━━━ 6s 113ms/step - accuracy: 0.4521 - loss: 0.7216 - val_accuracy: 0.5800 - val_loss: 0.7175
Epoch 7/20
56/56 ━━━━━━━━━━━━━━━━━━━━ 6s 113ms/step - accuracy: 0.4404 - loss: 0.7169 - val_accuracy: 0.4200 - val_loss: 0.7148
Epoch 8/20
56/56 ━━━━━━━━━━━━━━━━━━━━ 6s 112ms/step - accuracy: 0.4618 - loss: 0.714

In [41]:
save_results("hiv_ddi_100", accuracy_per_fold, auc_per_fold, all_classification_reports)
save_classification_report("hiv_ddi_100", avg_classification_report)

Results saved to reports/nosf/classification_results3_CNN.json
Classification report saved to reports/nosf/classification_reports3_cnn.json


In [42]:
accuracy_per_fold, auc_per_fold, all_classification_reports, avg_accuracy, std_accuracy, avg_auc, std_auc, avg_classification_report = cross_validate_model(image_folder_hivrtv, labels_file_hivrtv, num_folds=10)


Training fold 1...
Epoch 1/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 8s 112ms/step - accuracy: 0.4884 - loss: 0.9678 - val_accuracy: 0.5593 - val_loss: 0.7590
Epoch 2/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 7s 108ms/step - accuracy: 0.4812 - loss: 0.7554 - val_accuracy: 0.5593 - val_loss: 0.7383
Epoch 3/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 7s 109ms/step - accuracy: 0.5455 - loss: 0.7358 - val_accuracy: 0.5593 - val_loss: 0.7279
Epoch 4/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 7s 108ms/step - accuracy: 0.5396 - loss: 0.7266 - val_accuracy: 0.5593 - val_loss: 0.7211
Epoch 5/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 7s 109ms/step - accuracy: 0.5219 - loss: 0.7219 - val_accuracy: 0.5593 - val_loss: 0.7158
Epoch 6/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 7s 111ms/step - accuracy: 0.5150 - loss: 0.7163 - val_accuracy: 0.5593 - val_loss: 0.7122
Epoch 7/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 7s 109ms/step - accuracy: 0.5341 - loss: 0.7123 - val_accuracy: 0.5593 - val_loss: 0.7088
Epoch 8/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 7s 109ms/step - accuracy: 0.5268 - loss: 0.710

In [43]:
save_results("hiv_rtv_100", accuracy_per_fold, auc_per_fold, all_classification_reports)
save_classification_report("hiv_rtv_100", avg_classification_report)

Results saved to reports/nosf/classification_results3_CNN.json
Classification report saved to reports/nosf/classification_reports3_cnn.json
